In [1]:
! pip install transformers datasets torch torchaudio

In [2]:
from transformers import PreTrainedTokenizerBase
from datasets import load_dataset
from typing import Union, Optional
from transformers.utils import PaddingStrategy
import random

In [ ]:
! hf auth login

? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: 4BUM-KZO5

    Waiting for authorization...
Token is valid.
The token `oauth-emilymcf` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-emilymcf`
Note: This token will be refreshed automatically when it expires.


In [3]:
split = "train"

In [4]:

audio = load_dataset(
    "datapointai/text-to-speech-human-preferences-315k",
    "audio",
    split=split,
    token=True,
    columns=["audio_key", "audio"]
)

audio_by_audio_key = {a["audio_key"]: a["audio"] for a in audio}

pairs = load_dataset(
        "datapointai/text-to-speech-human-preferences-315k",
        "pairs",
        split=split,
        token=True,
        columns=["pair_key", "model_a", "model_b", "audio_a_key", "audio_b_key"]
    )

def fill_pairs(sample):
    audio_a = audio_by_audio_key[sample["audio_a_key"]]
    audio_b = audio_by_audio_key[sample["audio_b_key"]]
    if not audio_a or not audio_b:
        print("missign audio")
        return

    audio_a = audio_a.get_all_samples().data
    audio_b = audio_b.get_all_samples().data

    return {
        "audio_a": audio_a,
        "audio_b": audio_b,
        "pair_key": sample["pair_key"],
        "model_a": sample["model_a"],
        "model_b": sample["model_b"]
    }
filled_pairs = pairs.map(fill_pairs, remove_columns=["audio_a_key", "audio_b_key"])


README.md:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

data/audio/train-0000.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/audio/train-0000.parquet: downloading bytes:           |  0.00B            

data/audio/train-0001.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/audio/train-0001.parquet: downloading bytes:           |  0.00B            

data/audio/train-0002.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

... (26 outputs left out)

data/pairs/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28350 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3150 [00:00<?, ? examples/s]

Map:   0%|          | 0/28350 [00:00<?, ? examples/s]

In [5]:
filled_pairs.to_csv("pairs.csv", index=False)

Creating CSV from Arrow format:   0%|          | 0/29 [00:00<?, ?ba/s]

9250205

In [12]:
filled_pairs = load_dataset(
    "emilymcf/joined-human-preferences-tts-315k",
    split=split,
    token=True,
)

pairs_by_pair_key = {p["pair_key"]: p for p in filled_pairs}

responses = load_dataset(
    "datapointai/text-to-speech-human-preferences-315k",
    "responses",
    split=split,
    token=True,
    columns=["pair_key", "prompt_id", "annotator", "chosen"]
)

responses_by_annotator_id = {}
for response in responses:
  if response["annotator"] not in responses_by_annotator_id:
      responses_by_annotator_id[response["annotator"]] = []
  responses_by_annotator_id[response["annotator"]].append(response)

def fill_responses(sample):
  pair = pairs_by_pair_key[sample["pair_key"]]
  if not pair:
    print("missing pair")
    return

  chosen_audio = pair["audio_a"] if sample["chosen"] == 'a' else pair["audio_b"]
  rejected_audio = pair["audio_b"] if sample["chosen"] == 'a' else pair["audio_a"]

  return {
      "prompt_id": sample["prompt_id"],
      "annotator": sample["annotator"],
      "chosen_model": pair["model_a"] if sample["chosen"] == 'a' else pair["model_b"],
      "rejected_model": pair["model_b"] if sample["chosen"] == 'a' else pair["model_a"],
      "chosen_audio": chosen_audio,
      "rejected_audio": rejected_audio
  }

responses = responses.map(fill_responses)

responses.to_csv("responses.csv")

Map:   0%|          | 0/283500 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/284 [00:00<?, ?ba/s]

100392482